# Rubin DP2 Spatial Masking with SkyKatana (Local Execution via IVOA TAP): Survey Property Maps, Milky Way Dust Correction, and Bright Stars

**Environment:** Local `lsdb_env-py313` via External RSP TAP (`https://data.lsst.cloud/api/tap`) + ESA Gaia DR3 TAP (`https://gea.esac.esa.int/tap-server/tap`) — *to run inside the RSP on Gen3 Butler, see `01_DP2_Skykatana_Mask_RSP.ipynb`*  
**References:**
- **DP2 Survey Property Maps (`sp_maps`)**: https://dp2.lsst.io/products/maps/sp_maps.html
- **DP2 Tutorial 311.1 (Milky Way Dust Extinction)**: https://dp2.lsst.io/tutorials/notebook/311/notebook-311-1.html
- **DP2 Tutorial 204.2 (`the_monster` Reference Catalog)**: https://dp2.lsst.io/tutorials/notebook/204/notebook-204-2.html
- **SkyKatana Documentation**: https://skykatana.readthedocs.io/en/latest/

---

## How the `SkyKatana` (`SkyMaskPipe`) Pipeline Works in This Local TAP Context

[SkyKatana](https://skykatana.readthedocs.io/en/latest/) represents spatial survey masks as a **multi-stage boolean pipeline** (`SkyMaskPipe`) built on top of two-level `HealSparseMap` bitmaps. Every stage in this notebook mirrors `01_DP2_Skykatana_Mask_RSP.ipynb`, replacing internal RSP Gen3 Butler binary reads with external IVOA TAP queries:

1. **Two-tier HEALPix geometry (`order_cov` + `order_sparse`)**:
   - A coarse **coverage index** (`order_cov = 5`, `nside_coverage = 32`, ~3.36 deg$^2$ per tile) tracks *which regions of the sky contain data* in ~12 KB.
   - Fine **sparse child pixels** (`order_sparse = 15`, `nside_sparse = 32768`, $0.64''$ pixels) are allocated *only* inside active coverage tiles and stored **bit-packed** (`1 bit/pixel`, an $8\times$ memory reduction vs `bool` and $64\times$ vs `float64`).
2. **Named Positive & Negative Mask Stages (`*mask`)**:
   - **Positive stage (`propmask`)**: Reconstructed locally from `dp2.VisitDetector` 4-corner CCD polygons (`band`, `expTime`, `seeing`, $5\sigma$ `magLim`) + `ivoa.ObsCore` `lsst.deep_coadd` polygons (`calib_level = 3`) via RSP TAP — enforcing minimal exposure count, multi-band filter multiplicity ($\ge K$ bands), seeing, and **dust-corrected** inverse-variance coadd $5\sigma$ limiting magnitudes.
   - **Negative stages (`dustmask`, `starmask`, `boxmask`, `mwmask`)**: Pixels to *veto* — high Galactic extinction sightlines ($E(B-V) > E(B-V)_{\max}$ from `dp2.Object.ebv` / SFD98), bright-star diffraction halos whose radii $R(m_{\mathrm{LSST}})$ are computed via SkyKatana's `radfunction` contract from Gaia DR3 synthetic LSST AB magnitudes, saturated CCD bleed trails, and low Galactic latitudes.
3. **Streaming Boolean Combination (`mkp.combine`)**:
   - Evaluates $\texttt{mask} = \texttt{propmask} \setminus (\texttt{dustmask} \cup \texttt{starmask} \cup \texttt{boxmask} \cup \texttt{mwmask})$ **one coarse coverage row at a time**, automatically harmonizing stages built at different HEALPix orders (e.g. `mwmask` at order 9, `propmask` / `dustmask` at order 13, `starmask` at order 15) without ever materializing full-sky pixel arrays in RAM.
4. **Science Products (`plot`, `plot_moc`, `plot_fracmap`, `makerans`, `apply`, `moc_from_stage`, `write`)**:
   - Generates multi-scale WCS inspection plots, fractional detection (`fracdet`) weight maps, uniform random catalogs, fast catalog filtering on `dp2.Object` TAP queries, and IVOA MOC files for `lsdb` / HATS partition pruning — all within a bounded **< 4 GiB local memory footprint**.

### One-Time Local Setup (`lsdb_env-py313` + RSP Bearer Token)
Create an RSP token with **`read:tap`** scope at https://data.lsst.cloud (**Account → Security tokens**) and save it locally:
```bash
printf '%s\n' 'gt-YOUR_RSP_TOKEN_HERE' > ~/.rsp-tap.token
chmod 600 ~/.rsp-tap.token
```
Select the **`Python (lsdb_env-py313)`** kernel. *(If `~/.rsp-tap.token` is not set yet, this notebook automatically falls back to `SyntheticDP2TapService` so every cell executes offline out of the box.)*

In [ ]:
from functools import partial
import json
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

from astropy.coordinates import Angle, SkyCoord
import astropy.units as u
import healsparse as hsp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from skykatana import SkyMaskPipe

from dp2_skykatana_mask import (
    DEFAULT_GAIA_TAP_URL,
    DEFAULT_RSP_TAP_URL,
    DP2LocalTapMaskBuilder,
    EBVProvider,
    build_dust_extinction_mask,
    build_local_tap_star_mask,
    build_tap_ebv_provider,
    build_tap_property_mask,
    deredden_magnitude,
    discover_dp2_tap_schema,
    get_rubin_band_extinction_rx,
    load_config,
    make_skykatana_radfunction,
    open_rsp_tap_client,
    skykatana_lsst_radfunction,
)
from dp2_skykatana_mask.memory import MemoryGuard, configure_rsp_thread_env

try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except NameError:
    pass

# Cap OpenMP/BLAS thread pools to 2 threads and initialize 32 GiB-compatible MemoryGuard
configure_rsp_thread_env(max_threads=2)
guard = MemoryGuard(max_ram_gib=32.0)
print('Local memory guard status:', guard.snapshot().format_short())

## 1. Connect to RSP External TAP (`dp2.VisitDetector`, `ivoa.ObsCore`, `dp2.Object`) & Gaia DR3 TAP

Externally (`https://data.lsst.cloud/api/tap`), Rubin's IVOA TAP service exposes the relational equivalents needed to build every stage of our mask locally without Gen3 Butler:
- **`dp2.VisitDetector` + `dp2.Visit`**: Calibrated per-CCD sky corner polygons (`llcra..lrcdec`), filter `band`, `expTime`, PSF `seeing` (FWHM in arcseconds), and single-visit $5\sigma$ point-source `magLim`.
- **`ivoa.ObsCore`**: `lsst.deep_coadd` (`calib_level = 3`) STC-S `s_region` patch polygons.
- **`dp2.Object`**: Coadd object catalog including Tutorial 311.1's SFD98 `ebv` column and `ugrizy` `cModelMag` / `psfMag` photometry.
- **Public ESA Gaia DR3 TAP** (`https://gea.esac.esa.int/tap-server/tap`, zero login required): All-sky `gaiadr3.gaia_source` (`ra`, `dec`, `phot_g_mean_mag`, `bp_rp`), which `dp2_skykatana_mask` transforms into synthetic apparent LSST `ugrizy` AB magnitudes matching `the_monster`.

When run offline without `~/.rsp-tap.token` or `$RSP_ACCESS_TOKEN`, the cell below falls back to `SyntheticDP2TapService` (including realistic ECDFS dithered detector pointings, 240 bright stars, and a localized Galactic cirrus cloud reaching $E(B-V) \approx 0.29\text{ mag}$ in the NE corner) so every cell executes end-to-end.

In [ ]:
OUT_ROOT = REPO_ROOT / 'outputs' / 'local_tap_walkthrough'
OUT_ROOT.mkdir(parents=True, exist_ok=True)

pilot_cfg = load_config(REPO_ROOT / 'configs' / 'dp2_ecdfs_pilot.yaml')

tap_client = open_rsp_tap_client(
    tap_url=DEFAULT_RSP_TAP_URL,
    token_file=pilot_cfg.access_token_file,  # None checks ~/.rsp-tap.token & $RSP_ACCESS_TOKEN
    allow_synthetic_fallback=True,
    synthetic_kwargs={
        'center_ra_deg': pilot_cfg.region.ra_deg,
        'center_dec_deg': pilot_cfg.region.dec_deg,
        'radius_deg': pilot_cfg.region.radius_deg,
        'n_stars': 240,
    },
)
USING_SYNTHETIC = bool(tap_client.is_synthetic)
if USING_SYNTHETIC:
    # Order 13 (nside=8192, ~25.8" pixels) executes in ~2s locally; use 14 or 15 for live science!
    pilot_cfg.property_maps.order_sparse = 13
    pilot_cfg.dust.order_sparse = 13
    pilot_cfg.monster_stars.order_sparse = 13
    pilot_cfg.output.order_out = 13
    print('No ~/.rsp-tap.token detected -> using offline SyntheticDP2TapService (with NE cirrus cloud).')
else:
    pilot_cfg.property_maps.order_sparse = 13
    print(f'Connected to live RSP External TAP: {tap_client.tap_url} (backend={tap_client.backend_name})')

schema_info = discover_dp2_tap_schema(tap_client)
print('Discovered DP2 TAP tables:')
print(f'  - Visit table         : {schema_info.visit_table}')
print(f'  - VisitDetector table : {schema_info.visit_detector_table}')
print(f'  - Object table        : {schema_info.object_table}')
print(f'  - IVOA ObsCore present: {schema_info.has_obscore}')
print(f'  - Bright-star TAP     : {DEFAULT_GAIA_TAP_URL}')

## 2. Milky Way Dust Extinction Correction (DP2 Tutorial 311.1)

Parts of the Rubin DP2 footprint extend into regions with non-negligible Milky Way foreground dust (low Galactic latitude fields, Galactic plane/bulge outskirts, and high-latitude cirrus). Following **[DP2 Tutorial 311.1](https://dp2.lsst.io/tutorials/notebook/311/notebook-311-1.html)**:

1. **Bandpass Extinction Coefficients ($R_b$)**: Using a Cardelli, Clayton & Mathis (1989) / O'Donnell (1994) dust law with $R_V = 3.1$ integrated over the Rubin LSST bandpasses (`rubin_sim.phot_utils.DustValues(r_v=3.1).r_x`, or exact analytical CCM89 fallback locally), the per-band extinction in magnitudes is:
   $$A_b(\alpha, \delta) = R_b \times E(B - V)_{\mathrm{SFD98}}(\alpha, \delta)$$
2. **De-reddening Extragalactic Limiting Magnitudes (`propmask`)**: For extragalactic galaxies, SNe, or QSOs behind the Milky Way dust screen, a source of intrinsic magnitude $m_{b,\mathrm{int}}$ arrives with observed magnitude $m_{b,\mathrm{obs}} = m_{b,\mathrm{int}} + A_b$. Therefore, the **effective extragalactic $5\sigma$ coadd limiting magnitude** of a pixel is:
   $$m_{b,\mathrm{lim}}^{\mathrm{dered}}(\alpha, \delta) = m_{b,\mathrm{lim}}^{\mathrm{obs}}(\alpha, \delta) - R_b \, E(B - V)(\alpha, \delta)$$
3. **Explicit High-Extinction Veto (`dustmask`) & SFD98 Caveats**:
   - Tutorial 311.1 highlights two physical caveats of 2D SFD98 $E(B-V)$ maps:
     1. **Unreliable at very low Galactic latitudes ($|b| \lesssim 5^\circ$)** due to unremoved far-IR thermal emission sources and high optical depth.
     2. **Integrated total Galactic column**: SFD98 integrates dust to infinity, which *over-estimates* extinction for foreground Milky Way stars at $d \lesssim$ few hundred pc.
   - Consequently, our pipeline:
     - Queries `dp2.Object.ebv` via TAP (or local SFD98 maps for full-sky runs) and de-reddens coadd `maglim` when building `propmask`,
     - Adds a dedicated negative `dustmask` stage vetoing sightlines with $E(B-V)$ > `max_ebv` (`0.12` for WFD, `0.08` for strict cosmology, `0.20` in this pilot) or $|b|$ < `min_galactic_lat_deg`,
     - Keeps **bright-star magnitudes (`starmask`) as *observed* apparent LSST magnitudes**, because optical ghosting, scattered-light halos, and CCD saturation spikes depend strictly on the *observed* photon flux hitting LSSTCam!

In [ ]:
# 1. Retrieve Rubin R_b = A_b / E(B-V) coefficients
rx = get_rubin_band_extinction_rx(r_v=pilot_cfg.dust.r_v)
dust_df = pd.DataFrame({
    'band': list(rx.keys()),
    'R_b (A_b / E(B-V))': [round(rx[b], 3) for b in rx],
    'A_b at E(B-V)=0.05 [mag]': [round(0.05 * rx[b], 3) for b in rx],
    'A_b at E(B-V)=0.12 [mag]': [round(0.12 * rx[b], 3) for b in rx],
    'A_b at E(B-V)=0.25 [mag]': [round(0.25 * rx[b], 3) for b in rx],
})
display(dust_df)

# 2. Build the SFD98 E(B-V) provider from RSP TAP `dp2.Object.ebv` (or local SFD98 map)
ebv_provider = build_tap_ebv_provider(tap_client, pilot_cfg, schema_info=schema_info)
print('Active E(B-V) source:', ebv_provider.source_name)

In [ ]:
# Visualize E(B-V) and the resulting g-band depth loss A_g = R_g * E(B-V) across the pilot region
ra_c, dec_c, rad_deg = pilot_cfg.region.ra_deg, pilot_cfg.region.dec_deg, pilot_cfg.region.radius_deg
cos_d = np.cos(np.radians(dec_c))
ra_grid_1d = np.linspace(ra_c - rad_deg / cos_d, ra_c + rad_deg / cos_d, 140)
dec_grid_1d = np.linspace(dec_c - rad_deg, dec_c + rad_deg, 140)
ra_mesh, dec_mesh = np.meshgrid(ra_grid_1d, dec_grid_1d)

ebv_mesh = ebv_provider.eval_radec(ra_mesh.ravel(), dec_mesh.ravel()).reshape(ra_mesh.shape)
ag_mesh = rx['g'] * ebv_mesh

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8), constrained_layout=True)

im0 = axes[0].pcolormesh(ra_mesh, dec_mesh, ebv_mesh, shading='auto', cmap='magma')
axes[0].contour(ra_mesh, dec_mesh, ebv_mesh, levels=[pilot_cfg.dust.max_ebv], colors='cyan', linewidths=1.8)
axes[0].invert_xaxis()
axes[0].set_xlabel('RA [deg]')
axes[0].set_ylabel('Dec [deg]')
axes[0].set_title(f'Foreground Dust E(B-V) [cyan = max_ebv ({pilot_cfg.dust.max_ebv:.2f})]')
fig.colorbar(im0, ax=axes[0], label='E(B-V) [mag]')

im1 = axes[1].pcolormesh(ra_mesh, dec_mesh, ag_mesh, shading='auto', cmap='inferno')
axes[1].invert_xaxis()
axes[1].set_xlabel('RA [deg]')
axes[1].set_ylabel('Dec [deg]')
axes[1].set_title(f'LSST g-band Extinction A_g = {rx["g"]:.2f} * E(B-V) [mag]')
fig.colorbar(im1, ax=axes[1], label='A_g [mag]')
plt.show()

## 3. Step-by-Step Walkthrough of the `SkyMaskPipe` Pipeline (Local TAP)

Just as in `01_DP2_Skykatana_Mask_RSP.ipynb`, we instantiate a fresh `mkp = SkyMaskPipe(...)` object and build each stage explicitly to inspect how each cut transforms the survey area.

### Stage 3.1 — Survey Property Mask (`mkp.propmask`) via `dp2.VisitDetector` + `ivoa.ObsCore` With & Without Dust De-reddening

- **Coadd stacking from `dp2.VisitDetector`**: Each overlapping calibrated detector exposure $i$ contributes inverse-variance point-source weight $w_i \propto 10^{0.8\,m_{i,\mathrm{lim}}}$, yielding coadded $5\sigma$ depth
  $$m_{b,\mathrm{coadd}}^{\mathrm{obs}}(\alpha, \delta) = 1.25 \log_{10}\!\left(\sum_{i=1}^{N_b(\alpha,\delta)} 10^{0.8\, m_{i,\mathrm{lim}}}\right), \qquad m_{b,\mathrm{coadd}}^{\mathrm{dered}}(\alpha, \delta) = m_{b,\mathrm{coadd}}^{\mathrm{obs}}(\alpha, \delta) - R_b E(B-V)(\alpha, \delta),$$
  clipped to `ivoa.ObsCore` `lsst.deep_coadd` (`calib_level = 3`) polygons.
- **Full-DP2 scaling (`region.mode == 'full'`)**: `build_tap_property_mask` streams `dp2.VisitDetector` in $30^\circ$ RA stripes across $[0^\circ, 360^\circ)$ and evaluates **$K$-of-$N$ filter multiplicity** (`min_filters=4` + `required_bands=['r', 'i']`) directly into bit-packed `mkp.propmask` (`1 bit/pixel`).

Let us compare `propmask` **without** vs **with** the Tutorial 311.1 dust correction:

In [ ]:
import copy

# (A) Baseline SkyMaskPipe WITHOUT dust de-reddening of coadd maglim
cfg_nodust = copy.deepcopy(pilot_cfg)
cfg_nodust.dust.enabled = False
mkp_nodust = SkyMaskPipe(order_out=pilot_cfg.output.order_out, order_cov=pilot_cfg.output.order_cov)
_, stats_nodust = build_tap_property_mask(
    mkp_nodust, tap_client, cfg_nodust, guard, ebv_provider=None, schema_info=schema_info
)

# (B) Main SkyMaskPipe WITH Tutorial 311.1 dust de-reddening (m_lim_dered = m_lim_obs - R_b * E(B-V))
mkp = SkyMaskPipe(order_out=pilot_cfg.output.order_out, order_cov=pilot_cfg.output.order_cov)
_, prop_stats = build_tap_property_mask(
    mkp, tap_client, pilot_cfg, guard, ebv_provider=ebv_provider, rx_map=rx, schema_info=schema_info
)

delta_area = stats_nodust.area_deg2 - prop_stats.area_deg2
print(f'propmask valid area WITHOUT dust de-reddening : {stats_nodust.area_deg2:.4f} deg^2')
print(f'propmask valid area WITH    dust de-reddening : {prop_stats.area_deg2:.4f} deg^2')
print(f'Marginal depth area pruned by dust attenuation: {delta_area:.4f} deg^2 ({100.0 * delta_area / max(stats_nodust.area_deg2, 1e-9):.2f}%)')
print('\nPer-band surviving area (deg^2) after dust de-reddening:', prop_stats.per_band_area_deg2)

### Stage 3.2 — Explicit Dust Extinction & Galactic Latitude Veto (`mkp.dustmask` & `mkp.mwmask`)

Even where deeper coadd exposures compensate for $A_b$, high-$E(B-V)$ patches suffer from spatially variable extinction systematics ($\Delta R_V$, far-IR background confusion). `build_dust_extinction_mask` evaluates $E(B-V)$ over the active coverage tiles and attaches a bit-packed negative stage **`mkp.dustmask`** wherever $E(B-V)$ > `max_ebv` (or $|b|$ < `min_galactic_lat_deg`).

For wide-area surveys crossing the Milky Way, SkyKatana also provides `mkp.build_milkyway_mask(order_sparse=9, b0_deg=15.0, bulge_a_deg=25.0, bulge_b_deg=20.0, bit_packed=True)` to attach an analytical Galactic plane + bulge polygon stage (`mkp.mwmask`).

In [ ]:
_, dust_stats = build_dust_extinction_mask(
    mkp,
    footprint_mask=mkp.propmask,
    cfg=pilot_cfg,
    ebv_provider=ebv_provider,
    guard=guard,
    rx_map=rx,
)
print('Dust stage statistics:', json.dumps(dust_stats.__dict__, indent=2))

### Stage 3.3 — Bright-Star Masking (`mkp.starmask` & `mkp.boxmask`) Using Gaia DR3 TAP + SkyKatana's `radfunction` vs LSST Magnitude

On the RSP, `the_monster` (DMTN-277) combines Gaia DR3 positions with synthetic LSST fluxes derived from Gaia photometry/XP spectra. Locally, `build_local_tap_star_mask` queries public **ESA Gaia DR3 TAP** (`phot_g_mean_mag`, `bp_rp`; streamed over occupied sky cones for full-survey DP2 runs), transforms $(G, G_{\mathrm{BP}} - G_{\mathrm{RP}}) \to$ synthetic apparent LSST `ugrizy` AB magnitudes (`gaia_to_synthetic_lsst_magnitudes`), and invokes SkyKatana's **`radfunction(df, **kwargs)`** contract (`skykatana_lsst_radfunction`):

- **Single-band exponential law (`band='r'`, `'g'`, `'i'`, `'u'`, `'z'`, `'y'`)**:
  $$R_{\mathrm{arcsec}}(m_{\mathrm{LSST}}) = A_b \, \exp\!\left(B_b \, m_{\mathrm{LSST}}\right), \qquad \texttt{df['radius']} = \frac{R_{\mathrm{arcsec}}}{3600}$$
- **3-piece extended scattered-light + wing + core law (`band='r_halo'`)**:
  Captures wide-angle LSST optical ghosts and outer PSF wings for bright stars ($m_r < 13.67$) and transitions smoothly into the core profile ($m_r \ge 15.0$).
- **Multi-band envelope (`band='multi'`)**:
  Evaluates $R_b(m_b)$ across all synthetic LSST bands per star and adopts $\max_b R_b(m_b)$.

#### How `boxmask` (CCD Saturation Bleed Trails) & Orientation (`bleed_pa_deg`) Work
- **Dimensions**: For ultra-bright stars ($m_{\mathrm{LSST}} \le \texttt{bleed\_mag\_max}$, default `11.5`), a narrow rectangular bleed-trail mask is centered on $(\alpha_c, \delta_c)$ with angular width and length scaled from the star's halo radius $R(m_{\mathrm{LSST}})$:
  $$\text{width} = f_w \, R(m_{\mathrm{LSST}}) \; (\texttt{bleed\_width\_factor} = 0.25), \qquad \text{height} = f_h \, R(m_{\mathrm{LSST}}) \; (\texttt{bleed\_height\_factor} = 2.5).$$
- **Orientation (`bleed_pa_deg`) & Rubin `LSSTCam` `deepCoadd` physics**:
  - Upstream `SkyMaskPipe.pixelate_boxes` locks box orientation strictly to $\mathrm{PA} = 0^\circ$ (North–South) or $90^\circ$ (East–West, e.g. fixed-PA equatorial cameras) and foreshortens the major semi-axis by $\cos\delta$. Our `build_oriented_bleed_box_mask` computes exact 4-corner gnomonic tangent-plane rectangles rotated by **`monster_stars.bleed_pa_deg`** (degrees East of North: `0°` = N–S, `90°` = E–W, or any arbitrary camera `skyRotation` angle) and pixelizes them via `SkyMaskPipe.build_poly_mask`.
  - Because the Simonyi Survey Telescope has an **alt-azimuth mount** and `LSSTCam` actively dithers `skyRotation` across visits, single-visit CCD bleed trails rotate across position angles on the sky from exposure to exposure (and are largely rejected via `SAT`/`SUSPECT` mask planes during coaddition). Hence **`configs/dp2_wfd_default.yaml` sets `enable_bleed_boxes: false` by default** on multi-visit `deepCoadd`s, while `dp2_ecdfs_pilot.yaml` enables it (`bleed_pa_deg: 0.0`) to demonstrate the `boxmask` stage.

In [ ]:
# Demonstrate SkyKatana's partial-wrapped radfunction(df) across LSST magnitudes 4 <= m_LSST <= 18.5
mag_grid = np.linspace(4.0, 18.5, 300)
sample_df = pd.DataFrame({'mag': mag_grid, 'lsst_g_mag': mag_grid + 0.3, 'lsst_r_mag': mag_grid, 'lsst_i_mag': mag_grid - 0.2})

radfn_r85 = partial(skykatana_lsst_radfunction, band='r', completeness=0.85, min_radius_arcsec=0.0, max_radius_arcsec=3600.0)
radfn_r90 = partial(skykatana_lsst_radfunction, band='r', completeness=0.90, min_radius_arcsec=0.0, max_radius_arcsec=3600.0)
radfn_g85 = partial(skykatana_lsst_radfunction, band='g', completeness=0.85, min_radius_arcsec=0.0, max_radius_arcsec=3600.0)
radfn_halo = partial(skykatana_lsst_radfunction, band='r_halo', completeness=0.85, min_radius_arcsec=0.0, max_radius_arcsec=3600.0)

fig, ax = plt.subplots(figsize=(8.5, 4.8))
ax.semilogy(mag_grid, radfn_halo(sample_df.copy())['radius'] * 3600.0, lw=2.2, color='crimson', label='LSST r_halo (3-piece ghost + wing + core, c=85%)')
ax.semilogy(mag_grid, radfn_r90(sample_df.copy())['radius'] * 3600.0, lw=2.0, ls='--', color='darkorange', label='LSST r-band (strict c=90%)')
ax.semilogy(mag_grid, radfn_g85(sample_df.copy())['radius'] * 3600.0, lw=1.8, ls='-.', color='seagreen', label='LSST g-band (c=85%)')
ax.semilogy(mag_grid, radfn_r85(sample_df.copy())['radius'] * 3600.0, lw=2.0, color='royalblue', label='LSST r-band default (c=85%)')

ax.axvline(pilot_cfg.monster_stars.bleed_mag_max, color='gray', ls=':', lw=1.2, label=f'CCD bleed-box threshold (m <= {pilot_cfg.monster_stars.bleed_mag_max})')
ax.set_xlabel('Observed Synthetic LSST Magnitude (Gaia DR3 -> LSST) [AB mag]')
ax.set_ylabel('Masking Halo Radius R(m_LSST) [arcsec]')
ax.set_title('SkyKatana Stellar Mask Radius Function vs. LSST Magnitude')
ax.grid(True, which='both', alpha=0.3)
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Build the active SkyKatana radfunction and stream bright stars via Gaia DR3 TAP
active_radfn = make_skykatana_radfunction(pilot_cfg.monster_stars)

star_stats = build_local_tap_star_mask(
    mkp,
    footprint_mask=mkp.propmask,
    cfg=pilot_cfg,
    guard=guard,
    work_dir=OUT_ROOT / 'intermediate',
    rsp_tap_client=tap_client,
    radfunction=active_radfn,
)
print('Local TAP bright-star masking summary:', json.dumps(star_stats.__dict__, indent=2))

### Stage 3.4 — Streaming Boolean Combination (`mkp.combine`) & Pipeline Introspection (`print(mkp)`, `stage_meta`)

Now `mkp` holds our positive footprint stage (`propmask`) and negative veto stages (`dustmask`, `starmask`, `boxmask`, plus optional per-band `prop_{b}_mask` stages). Calling `mkp.combine(positive=..., negative=..., bit_packed=True)` streams coverage row-by-row to compute:
$$\texttt{mask} = \texttt{propmask} \setminus \left(\texttt{dustmask} \cup \texttt{starmask} \cup \texttt{boxmask}\right)$$

Printing `mkp` invokes SkyKatana's dynamic stage discovery (`SkyMaskPipe.__str__`), listing every attached `*mask` stage, its `(order/nside)` geometry, valid pixel count, sky area, and pixel scale!

In [ ]:
neg_stages = []
for st_name in ('dustmask', 'starmask', 'boxmask', 'mwmask'):
    st_map = getattr(mkp, st_name, None)
    if st_map is not None and int(st_map.n_valid) > 0:
        neg_stages.append(st_name)

combined_mask = mkp.combine(
    positive=['propmask'],
    negative=neg_stages,
    order_out=pilot_cfg.output.order_out,
    order_cov=pilot_cfg.output.order_cov,
    bit_packed=True,
    verbose=True,
)

print('\n=== SkyMaskPipe Stage Table (print(mkp)) ===')
print(mkp)

print('\n=== Stored Stage Provenance (mkp.stage_meta("propmask")) ===')
print(json.dumps(mkp.stage_meta('propmask'), indent=2))

### Stage 3.5 — Visualizing Stages with SkyKatana's Native Plotting Suite (`plot`, `plot_moc`, `plot_fracmap`)

SkyKatana includes three complementary visualization methods on `SkyMaskPipe`:
1. **`mkp.plot(stage='mask', plot_circles=..., ax=ax)`**: Samples uniform random points inside `stage` and overlays analytic circle boundaries from the bright-star table so you can verify sub-arcsecond star-hole alignment.
2. **`mkp.plot_moc(stage, center=..., fov=..., ax=..., wcs=...)`**: Builds multi-order HEALPix coverage cells on a celestial `mocpy.WCS` projection and returns `(fig, ax, wcs)` so multiple stages (`propmask`, `dustmask`, `starmask`) can be layered on the exact same sky axes.
3. **`mkp.plot_fracmap(stage, order_frac=..., center=..., fov=..., thresholds=...)`**: Computes `SkyMaskPipe.frac_area_map` (fractional unmasked pixel coverage $f \in [0, 1]$ at coarse HEALPix order `order_frac`, used for angular clustering weights) and renders it with isocontours.

In [ ]:
ra_min = ra_c - rad_deg / cos_d
ra_max = ra_c + rad_deg / cos_d
dec_min = dec_c - rad_deg
dec_max = dec_c + rad_deg
center_sc = SkyCoord(ra=ra_c * u.deg, dec=dec_c * u.deg, frame='icrs')
fov_2d = [Angle(2.25 * rad_deg, 'deg'), Angle(2.25 * rad_deg, 'deg')]

# 1. Zoomed SkyMaskPipe.plot() with analytical R(m_LSST) star circles & bleed boxes overlaid
zoom_dra = 0.35 / cos_d
zoom_ddec = 0.35
boxes_pilot_path = OUT_ROOT / 'intermediate' / 'monster_bleed_boxes.fits'
fig1, ax1 = plt.subplots(figsize=(7.2, 6.4))
mkp.plot(
    stage='mask',
    nr=40_000,
    s=0.75,
    clipra=(ra_c - zoom_dra, ra_c + zoom_dra),
    clipdec=(dec_c - zoom_ddec, dec_c + zoom_ddec),
    plot_circles={
        'data': star_stats.circles_parquet_path,
        'fmt': 'parquet',
        'columns': ['ra', 'dec', 'radius'],
    },
    plot_boxes=(
        {'data': str(boxes_pilot_path), 'fmt': 'fits', 'columns': ['ra_c', 'dec_c', 'width', 'height']}
        if boxes_pilot_path.is_file()
        else False
    ),
    ax=ax1,
    color='#0f766e',
    alpha=0.68,
)
ax1.invert_xaxis()  # Astronomical East-left convention applied after clipra filtering
ax1.set_title('Zoomed SkyMaskPipe.plot("mask") + R(m_LSST) Circles & Bleed Boxes')
plt.tight_layout()
plt.show()

# 2. Layered WCS MOC view: propmask (slate) - dustmask (orange) - starmask (crimson) -> final mask (teal)
fig2, ax2, wcs2 = mkp.plot_moc(
    'propmask',
    center=center_sc,
    fov=fov_2d,
    color='#94a3b8',
    alpha=0.35,
    linewidth=0.0,
    label='propmask (dust-corrected depth)',
    figsize=(7.6, 7.0),
    show=False,
)
if getattr(mkp, 'dustmask', None) is not None and int(mkp.dustmask.n_valid) > 0:
    mkp.plot_moc(
        'dustmask',
        ax=ax2,
        wcs=wcs2,
        color='#f97316',
        alpha=0.70,
        linewidth=0.0,
        label=f'dustmask (E(B-V) > {pilot_cfg.dust.max_ebv})',
        show=False,
    )
mkp.plot_moc(
    'mask',
    ax=ax2,
    wcs=wcs2,
    color='#0f766e',
    alpha=0.85,
    linewidth=0.0,
    label='final combined mask',
    show=False,
)
if getattr(mkp, 'starmask', None) is not None and int(mkp.starmask.n_valid) > 0:
    mkp.plot_moc(
        'starmask',
        ax=ax2,
        wcs=wcs2,
        color='#dc2626',
        alpha=0.75,
        linewidth=0.0,
        label='starmask (LSST halos)',
        show=False,
    )
ax2.set_title('Layered SkyMaskPipe.plot_moc: Footprint, Vetoes & Surviving Mask', pad=10)
ax2.legend(loc='upper right', fontsize=8.5, framealpha=0.92)
plt.show()

# 3. Coarse fractional coverage map (SkyMaskPipe.plot_fracmap, order=8 -> ~13.7' cells) for LSS weights
frac_order = min(int(pilot_cfg.output.fracdet_order), pilot_cfg.output.order_out - 2)
fig3, ax3, wcs3, im3, _ = mkp.plot_fracmap(
    'mask',
    order_frac=frac_order,
    center=center_sc,
    fov=fov_2d,
    vmin=0.0,
    vmax=1.0,
    cmap='Spectral_r',
    thresholds=[0.5, 0.9],
    contour_kwargs={'colors': 'black', 'linewidths': 0.55, 'alpha': 0.55},
    figsize=(8.2, 7.0),
)
ax3.set_title(f'SkyMaskPipe.plot_fracmap("mask", order={frac_order})', pad=12)
plt.show()

### Stage 3.6 — Randoms (`makerans`), Filtering `dp2.Object` Galaxies via TAP (`apply`), IVOA MOC (`moc_from_stage`), and Persistence (`write` / `read`)

Once `mkp.mask` is built, SkyKatana supports four immediate science operations locally:
1. **`mkp.makerans(stage='mask', nr=..., rng=...)`**: Two-pass multinomial streaming random generator.
2. **`mkp.apply(stage='mask', cat=..., columns=('coord_ra', 'coord_dec'))`**: Filters galaxies queried directly from **`dp2.Object` via RSP TAP** (or a lazy `lsdb` HATS catalog).
3. **`SkyMaskPipe.moc_from_stage(mkp.mask, order_forced=...)`**: Streams coverage rows to construct a compact `mocpy.MOC` for server-side `lsdb.MOCSearch` partition pruning.
4. **`mkp.write(dir)` / `SkyMaskPipe.read(dir)`**: Streams per-row bit-packed FITS stage files + `metadata.json` to disk and reloads them on demand.

In [ ]:
# 1. Generate 20,000 uniform random points strictly inside the surviving mask
rans = mkp.makerans(stage='mask', nr=20_000, rng=np.random.RandomState(2026))

# 2. Persist the entire multi-stage SkyMaskPipe and reload it from disk
pipe_dir = OUT_ROOT / 'skykatana_pipe_manual'
mkp.write(pipe_dir, overwrite=True)
reloaded_mkp = SkyMaskPipe.read(pipe_dir, verbose=False)

# 3. Query extragalactic galaxies (`refExtendedness = 1`) from `dp2.Object` via TAP and filter with `.apply()`!
adql_gal = (
    f"SELECT TOP 18000 objectId, coord_ra, coord_dec, ebv, g_cModelMag, r_cModelMag, i_cModelMag "
    f"FROM {schema_info.object_table} "
    f"WHERE CONTAINS(POINT('ICRS', coord_ra, coord_dec), "
    f"CIRCLE('ICRS', {ra_c:.6f}, {dec_c:.6f}, {rad_deg:.6f})) = 1 "
    f"AND refExtendedness = 1"
)
tap_galaxies = tap_client.search(adql_gal, maxrec=18_000).to_table().to_pandas()
clean_galaxies = reloaded_mkp.apply(stage='mask', cat=tap_galaxies, columns=('coord_ra', 'coord_dec')).copy()

# De-redden surviving galaxies' photometry using Tutorial 311.1's dust correction!
clean_ebv = clean_galaxies['ebv'].to_numpy(dtype=float)
clean_galaxies['g_cModelMag_dered'] = deredden_magnitude(clean_galaxies['g_cModelMag'].to_numpy(dtype=float), clean_ebv, 'g', rx_map=rx)
clean_galaxies['r_cModelMag_dered'] = deredden_magnitude(clean_galaxies['r_cModelMag'].to_numpy(dtype=float), clean_ebv, 'r', rx_map=rx)

# 4. Export an IVOA MOC for lsdb.read_hats(..., search_filter=lsdb.MOCSearch(moc))
moc = SkyMaskPipe.moc_from_stage(reloaded_mkp.mask, order_forced=min(12, pilot_cfg.output.order_out), progress=False)
moc_path = OUT_ROOT / 'ecdfs_pilot_moc.fits'
moc.save(str(moc_path), format='fits', overwrite=True)

print(f'Randoms generated inside mask  : {len(rans):,}')
print(f'Queried TAP galaxies           : {len(tap_galaxies):,} -> Surviving clean mask: {len(clean_galaxies):,}')
print(f'Max E(B-V) among clean galaxies: {clean_galaxies["ebv"].max():.4f} mag (<= max_ebv={pilot_cfg.dust.max_ebv})')
print(f'Saved IVOA MOC to              : {moc_path} (sky_fraction={moc.sky_fraction:.3e})')
display(clean_galaxies[['objectId', 'coord_ra', 'coord_dec', 'ebv', 'g_cModelMag', 'g_cModelMag_dered']].head())

## 4. Automated Pipeline (`DP2LocalTapMaskBuilder`) & Building the Complete Multi-Band DP2 Mask Locally via `run_local_tap_mask.sh`

`DP2LocalTapMaskBuilder` is the exact local-TAP counterpart to `DP2MaskBuilder` — encapsulating every step demonstrated in Section 3 (`propmask` stacked across all 6 `ugrizy` bands from `dp2.VisitDetector` with `min_filters=4`, `required_bands=['r', 'i']`, and Tutorial 311.1 dust de-reddening, plus `dustmask`, `mwmask`, `starmask` from Gaia DR3 TAP via `skykatana_lsst_radfunction`, `combine`, `makerans`, coarse `fracdet` at `order=6`, IVOA MOC export, QA plots, and `mask_build_summary.json`).

You can construct the **complete full-survey DP2 mask** (`configs/dp2_wfd_default.yaml`) **locally right from the cell below** via `run_mask_script(REPO_ROOT / 'scripts' / 'run_local_tap_mask.sh', ...)` — which runs `./scripts/run_local_tap_mask.sh` in a clean child OS process so all temporary stacking buffers are freed before Section 4.1 loads `outputs/dp2_wfd_default/skykatana_pipe` — or launch the exact same script inside `tmux` in your local terminal:

```bash
# Ensure ~/.rsp-tap.token (with `read:tap` scope) is present for live DP2 TAP, then run:
tmux new -s dp2tap
./scripts/run_local_tap_mask.sh configs/dp2_wfd_default.yaml
```

In [ ]:
from dp2_skykatana_mask import run_mask_script

pilot_cfg.output.output_dir = str(REPO_ROOT / 'outputs' / 'local_tap_ecdfs_pilot')
builder = DP2LocalTapMaskBuilder(pilot_cfg, tap_client=tap_client)

# 1. Estimate full-DP2 (6,500 deg^2 at order=15) memory footprint in Local TAP mode before executing
wfd_cfg = load_config(REPO_ROOT / 'configs' / 'dp2_wfd_default.yaml')
print('Full-DP2 Local TAP (6,500 deg^2, order_out=15) RAM budget (GiB):',
      json.dumps(DP2LocalTapMaskBuilder(wfd_cfg).estimate_memory(assumed_area_deg2=6500.0), indent=2))

# 2. Run automated ECDFS pilot in-process
auto_pipe, summary = builder.run()
print(f"\nAutomated local TAP pilot mask built: {summary.final_area_deg2:.3f} deg^2 (peak RAM = {summary.peak_ram_gib:.2f} GiB)")

# 3. Construct the COMPLETE multi-band DP2 mask (`configs/dp2_wfd_default.yaml`) locally via `scripts/run_local_tap_mask.sh`!
BUILD_COMPLETE_DP2_MASK = True
if BUILD_COMPLETE_DP2_MASK:
    print('\n=== Executing ./scripts/run_local_tap_mask.sh configs/dp2_wfd_default.yaml ===')
    wfd_summary = run_mask_script(
        REPO_ROOT / 'scripts' / 'run_local_tap_mask.sh',
        REPO_ROOT / 'configs' / 'dp2_wfd_default.yaml',
        extra_args=(['--allow-synthetic-fallback'] if USING_SYNTHETIC else []),
    )
    print(
        f"\nComplete DP2 mask built locally in {wfd_summary.get('output_files', {}).get('skykatana_pipe_dir')}: "
        f"surviving area = {wfd_summary.get('final_area_deg2', 0.0):,.2f} deg^2, "
        f"peak RAM = {wfd_summary.get('peak_ram_gib', 0.0):.2f} GiB"
    )

### 4.1 Multi-Scale Visualization of the Complete Combined Mask (`stage='mask'`) with SkyKatana

Identical to Section 4.1 of `01_DP2_Skykatana_Mask_RSP.ipynb`, a complete Rubin DP2 mask at HEALPix order 15 ($0.64''$ pixels) spans **five orders of magnitude** in angular scale — from $10''$–$150''$ star-halo holes up to $100^\circ+$ survey strips crossing the $\mathrm{RA} = 0^\circ / 360^\circ$ wrap.

`dp2_skykatana_mask` renders the complete mask with three memory-safe helpers built directly on SkyKatana's native visualization methods:
1. **`plot_complete_mask_multiscale(full_pipe, ...)`**:
   - **Left panel (`SkyMaskPipe.plot_moc`)**: Tight 2D `[fov_lon, fov_lat]` WCS framing — tangent `SIN` for regional masks, or equatorial `CAR` (`CRVAL2 = 0`) cropped to the occupied declination band for wide surveys so $\mathrm{RA} = 0^\circ / 360^\circ$ never splits the footprint — layering `propmask`, `dustmask`, `mwmask`, `starmask`, and the surviving `mask` cleanly without internal HEALPix cell seams.
   - **Right panel (`SkyMaskPipe.pix_in_zone` + `SkyMaskPipe.plot`)**: Automatically selects a high-occupancy interior tile containing prominent bright stars, extracts a $<1\,\mathrm{MB}$ sub-map via `pix_in_zone`, and renders 45,000 uniform random points with analytical `R(m_LSST)` circles (`plot_circles`) and CCD bleed boxes (`plot_boxes`).
2. **`plot_complete_mask_fracmap(full_pipe, ...)`**: High-contrast fractional coverage $f \in [0, 1]$ (`SkyMaskPipe.plot_fracmap`, `cmap='Spectral_r'`) exposing survey border roll-off, dust cavities, and star-halo area dilution.
3. **`plot_complete_mask_allsky(full_pipe)`**: Full $4\pi$ equatorial Mollweide (`projection='MOL'`) context (`SkyMaskPipe.plot_moc` + `SkyMaskPipe.gal_plane_bulge_moc` + `SkyMaskPipe.plot_srcs`).

*(If you have already built the full-survey DP2 mask locally via `./scripts/run_local_tap_mask.sh configs/dp2_wfd_default.yaml` — or copied `outputs/dp2_wfd_default/skykatana_pipe` from the RSP — the cell below loads it automatically; otherwise it visualizes `auto_pipe` built above.)*

In [ ]:
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except NameError:
    pass

from dp2_skykatana_mask import (
    compute_sky_extent,
    plot_complete_mask_allsky,
    plot_complete_mask_fracmap,
    plot_complete_mask_multiscale,
)

# Load full-survey DP2 WFD pipeline if already built locally (or synced from RSP); otherwise use `auto_pipe`
wfd_pipe_dir = REPO_ROOT / 'outputs' / 'dp2_wfd_default' / 'skykatana_pipe'
if (wfd_pipe_dir / 'metadata.json').is_file():
    print(f'Loading full-survey DP2 SkyMaskPipe from: {wfd_pipe_dir}')
    full_pipe = SkyMaskPipe.read(wfd_pipe_dir, verbose=False)
    full_circles_path = wfd_pipe_dir.parent / 'monster_bright_stars_circles.parquet'
    full_boxes_path = wfd_pipe_dir.parent / 'monster_bleed_boxes.fits'
else:
    full_pipe = auto_pipe
    _circ_raw = getattr(summary.monster_stars, 'circles_parquet_path', None)
    full_circles_path = Path(_circ_raw) if _circ_raw else None
    full_boxes_path = Path(pilot_cfg.output.output_dir) / 'monster_bleed_boxes.fits'

sky_ext = compute_sky_extent(full_pipe, 'mask')
print('=== Active SkyMaskPipe Summary ===')
print(full_pipe)
print(
    f"Sky extent: center=({sky_ext['ra_c']:.2f}°, {sky_ext['dec_c']:.2f}°), "
    f"ΔRA={sky_ext['ra_span']:.2f}°, Dec=[{sky_ext['dec_min']:.2f}°, {sky_ext['dec_max']:.2f}°], "
    f"wide_survey={sky_ext['wide_survey']}"
)

# 1. Complete Mask Footprint (SkyMaskPipe.plot_moc) + Interior Star-Halo & Bleed Cutout (pix_in_zone + plot)
#    Tip: pass `zoom_center=(ra_deg, dec_deg)` to inspect any specific sky coordinates!
fig_ms, (ax_moc_ov, ax_zoom) = plot_complete_mask_multiscale(
    full_pipe,
    circles_path=full_circles_path,
    boxes_path=full_boxes_path,
    zoom_half_width_deg=0.25,
)

# 2. Coarse Fractional Coverage f in [0, 1] (order=6 -> ~55' cells for wide DP2; order=8 -> ~13.7' for pilot)
frac_map_order = 6 if sky_ext['wide_survey'] else 8
fig_frac, ax_frac, wcs_frac = plot_complete_mask_fracmap(full_pipe, order_frac=frac_map_order)

# 3. Full 4pi Equatorial Mollweide Context (SkyMaskPipe.plot_moc + Galactic Plane + plot_srcs)
fig_mol, ax_mol, wcs_mol = plot_complete_mask_allsky(full_pipe)

# Optional interactive Aladin Lite pan-and-zoom widget in JupyterLab (requires `ipyaladin`):
# al = full_pipe.plot_moca('mask', color='teal', opacity=0.45, target=f"{sky_ext['ra_c']:.3f} {sky_ext['dec_c']:.3f}", fov=max(sky_ext['ra_span'], 3.0), order_forced=min(12, full_pipe.order_out))

### Q&A

- **How does local TAP execution compare with running inside the RSP?** Inside the RSP (`01_DP2_Skykatana_Mask_RSP.ipynb`), `DP2MaskBuilder` reads Gen3 Butler `HealSparseMap` `sp_maps` and `htm7` `the_monster` shards directly. Locally (`02_DP2_Skykatana_Mask_Local_TAP.ipynb`), `DP2LocalTapMaskBuilder` stacks calibrated `dp2.VisitDetector` polygons (+ `ivoa.ObsCore` coadd patches) and `dp2.Object.ebv` from `https://data.lsst.cloud/api/tap` (`read:tap` Bearer token) and pairs them with public ESA Gaia DR3 TAP (`phot_g_mean_mag`, `bp_rp` $\to$ synthetic LSST `ugrizy` AB magnitudes) to run the exact same `SkyMaskPipe` stages without `lsst_distrib`.
- **How is Milky Way dust extinction from DP2 Tutorial 311.1 handled locally?** Per-pixel inverse-variance coadd $5\sigma$ limiting magnitudes are de-reddened via $m_{b,\mathrm{lim}}^{\mathrm{dered}} = m_{b,\mathrm{lim}}^{\mathrm{obs}} - R_b E(B-V)$ using Rubin $R_V = 3.1$ coefficients, while `dustmask` vetoes sightlines exceeding `max_ebv` or below `min_galactic_lat_deg`.
- **How is `boxmask` computed and oriented?** `build_oriented_bleed_box_mask` constructs tangent-plane rectangles of width $0.25\,R(m_{\mathrm{LSST}})$ and height $2.5\,R(m_{\mathrm{LSST}})$ rotated by `bleed_pa_deg` East of North (`0°` = N–S, `90°` = E–W, or custom rotator angle); on multi-visit Rubin `deepCoadd`s (`dp2_wfd_default.yaml`), `enable_bleed_boxes` defaults to `false` because LSSTCam's alt-az camera rotator dithers `skyRotation` across visits.

### Data Analysis Key Findings

- Stacking `dp2.VisitDetector` exposures with inverse-variance weights $w_i \propto 10^{0.8\,m_{i,\mathrm{lim}}}$ reproduces spatially varying coadd depth roll-off across dithered focal-plane borders, and de-reddening by $A_b = R_b E(B-V)$ prunes marginal dust-attenuated pixels before `dustmask` excises the high-extinction cirrus core.
- Streaming $30^\circ$ RA stripes of `dp2.VisitDetector` at order 13 ($25.8''$ pixels) while pixelizing bright-star halos at order 15 ($0.64''$ pixels) and combining via `SkyMaskPipe.combine(..., bit_packed=True)` keeps full-survey 6,500 deg$^2$ DP2 local RAM usage within ~3–4 GiB.

### Insights or Next Steps

- Save your RSP `read:tap` token to `~/.rsp-tap.token` (`chmod 600 ~/.rsp-tap.token`), validate cuts on `configs/dp2_ecdfs_pilot.yaml`, and launch `./scripts/run_local_tap_mask.sh configs/dp2_wfd_default.yaml` inside `tmux` to build the complete DP2 mask locally (then re-run Section 4.1 above to inspect it!).
- Pair the exported `dp2_combined_mask_moc.fits` (`lsdb.MOCSearch`) with `SkyMaskPipe.apply(stage='mask', ...)` and `deredden_magnitude(...)` when analyzing DP2 TAP or HATS catalogs locally in `lsdb_env-py313`.